# DSP Foundations Examples

Explore the Phase 4 time-domain and frequency-domain helpers using synthetic signals. The examples keep the data generation explicit so each metric can be interpreted against a known signal.

## Setup

In [ ]:
from pathlib import Path
import sys

repo_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
src_path = repo_root / "src"
if src_path.exists() and str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

from signal_processing_prep.features import FeatureExtractor, FrequencyBand, SlidingWindowConfig
from signal_processing_prep.frequency_domain import (
    band_energy,
    dominant_frequency,
    fft_magnitude,
    psd,
    spectral_bandwidth,
    spectral_centroid,
    spectral_flatness,
    spectral_rolloff,
)
from signal_processing_prep.synthetic import (
    add_signals,
    chirp_signal,
    impulse_train,
    multiply_signals,
    noisy_sine_wave,
    sinc_signal,
    sine_wave,
    transient_burst,
    window_signal,
)
from signal_processing_prep.time_frequency import (
    hilbert_analysis,
    morlet_wavelet_scalogram,
    spectrogram_analysis,
    stft_analysis,
)
from signal_processing_prep.time_domain import (
    crest_factor,
    kurtosis,
    rms,
    skewness,
    zero_crossing_rate,
)

print(f"Using repository root: {repo_root}")

## Interactive Plot Backend

Run this before plotting if you want the standard matplotlib toolbar for zooming, panning, and saving figures.

In [ ]:
from IPython import get_ipython

ip = get_ipython()
if ip is not None:
    try:
        ip.run_line_magic("matplotlib", "widget")
        print("Using matplotlib widget backend.")
    except Exception:
        ip.run_line_magic("matplotlib", "inline")
        print("Widget backend unavailable; using inline plots.")


## Example Signals

In [ ]:
sampling_rate_hz = 2000.0
duration_seconds = 4.0

base_sine = sine_wave(
    frequency_hz=50.0,
    duration_seconds=duration_seconds,
    sampling_rate_hz=sampling_rate_hz,
    amplitude=1.0,
    name="50_hz_sine",
)
noisy_sine = noisy_sine_wave(
    frequency_hz=50.0,
    duration_seconds=duration_seconds,
    sampling_rate_hz=sampling_rate_hz,
    amplitude=1.0,
    noise_std=0.25,
    seed=7,
    name="50_hz_noisy_sine",
)
two_tone = add_signals(
    [
        sine_wave(
            frequency_hz=50.0,
            duration_seconds=duration_seconds,
            sampling_rate_hz=sampling_rate_hz,
            amplitude=1.0,
            name="50_hz_component",
        ),
        sine_wave(
            frequency_hz=180.0,
            duration_seconds=duration_seconds,
            sampling_rate_hz=sampling_rate_hz,
            amplitude=0.35,
            name="180_hz_component",
        ),
    ],
    label="custom",
    name="50_180_hz_two_tone",
)
impulses = impulse_train(
    duration_seconds=duration_seconds,
    sampling_rate_hz=sampling_rate_hz,
    impulse_rate_hz=20.0,
    amplitude=1.0,
    name="20_hz_impulse_train",
)
burst = transient_burst(
    duration_seconds=duration_seconds,
    sampling_rate_hz=sampling_rate_hz,
    burst_frequency_hz=450.0,
    burst_start_seconds=0.8,
    burst_duration_seconds=0.08,
    amplitude=1.0,
    name="450_hz_transient_burst",
)
window = window_signal(
    duration_seconds=duration_seconds,
    sampling_rate_hz=sampling_rate_hz,
    window_start_seconds=0.5,
    window_duration_seconds=0.75,
    window_type="hann",
    name="hann_window_0p5_to_1p25s",
)
sinc = sinc_signal(
    duration_seconds=duration_seconds,
    sampling_rate_hz=sampling_rate_hz,
    bandwidth_hz=80.0,
    center_seconds=1.0,
    name="80_hz_sinc",
)

records = [base_sine, noisy_sine, two_tone, impulses, burst, window, sinc]
[(record.name, record.n_samples, record.duration_seconds, record.sampling_rate_hz) for record in records]

## Time-Domain Metrics

RMS and crest factor provide compact amplitude summaries. Crest Factor (Scheitelfaktor): largest peak compared to RMS. Kurtosis is more statistically sensitive. Zero-crossing rate is useful for rough frequency or texture comparisons. Skewness and kurtosis capture distribution shape of waveform.

In [ ]:
import pandas as pd

time_domain_rows = []
for record in records:
    time_domain_rows.append(
        {
            "name": record.name,
            "label": record.label,
            "rms": rms(record),
            "crest_factor": crest_factor(record),
            "zero_crossings_per_second": zero_crossing_rate(record),
            "skewness": skewness(record),
            "kurtosis": kurtosis(record),
        }
    )

time_domain_df = pd.DataFrame(time_domain_rows)
time_domain_df

## Time-Domain Inspection

In [ ]:
from signal_processing_prep.plotting import plot_time_signal

for record in [base_sine, two_tone, burst, window, sinc]:
    plot_time_signal(record, start_seconds=0.0, duration_seconds=1.5, max_points=2500)

## Time-Localized Feature Dynamics

Global metrics compress the whole record into one value. Sliding-window features preserve temporal structure by computing the same metrics over local windows.

In [ ]:
localized_config = SlidingWindowConfig(
    window_seconds=0.2,
    step_seconds=0.04,
    include_frequency_domain=True,
    frequency_bands=(
        FrequencyBand("low_0_100_hz", 0.0, 100.0),
        FrequencyBand("high_300_600_hz", 300.0, 600.0),
    ),
)

localized_burst_features = FeatureExtractor().extract_windows(burst, localized_config).to_dataframe()
localized_burst_features.head()

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(3, 1, figsize=(10, 7), sharex=True)
axes[0].plot(localized_burst_features["window_center_seconds"], localized_burst_features["rms"])
axes[0].set_ylabel("RMS")
axes[0].set_title("Sliding-window feature dynamics for transient burst")

axes[1].plot(localized_burst_features["window_center_seconds"], localized_burst_features["skewness"])
axes[1].set_ylabel("Skewness")

axes[2].plot(
    localized_burst_features["window_center_seconds"],
    localized_burst_features["band_energy_high_300_600_hz"],
)
axes[2].set_ylabel("High-band energy")
axes[2].set_xlabel("Window center [s]")

for ax in axes:
    ax.grid(True, alpha=0.3)
fig.tight_layout()

## Window Choice And Spectral Leakage

Sliding-window frequency features use finite signal segments, so discontinuities at segment edges can leak energy into neighboring FFT bins. A taper such as Hann usually reduces sidelobes, while a rectangular window preserves raw amplitudes but leaks more when the tone is not bin-centered.

In [ ]:
leakage_record = sine_wave(
    frequency_hz=53.0,
    duration_seconds=3.0,
    sampling_rate_hz=2000.0,
    name="53_hz_non_bin_centered_sine",
)

rectangular_features = FeatureExtractor().extract_windows(
    leakage_record,
    SlidingWindowConfig(
        window_seconds=0.5,
        step_seconds=0.25,
        frequency_window=None,
        include_time_domain=False,
    ),
).to_dataframe()
hann_features = FeatureExtractor().extract_windows(
    leakage_record,
    SlidingWindowConfig(
        window_seconds=0.5,
        step_seconds=0.25,
        frequency_window="hann",
        include_time_domain=False,
    ),
).to_dataframe()

pd.concat(
    [
        rectangular_features.assign(window="rectangular"),
        hann_features.assign(window="hann"),
    ],
    ignore_index=True,
)[
    ["window", "window_center_seconds", "dominant_frequency_hz", "spectral_bandwidth_hz", "spectral_flatness"]
]

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import signal as scipy_signal
from signal_processing_prep.frequency_domain import fft_magnitude

window_seconds = 0.75
step_seconds = 0.25
sampling_rate_hz = leakage_record.sampling_rate_hz

window_samples = int(round(window_seconds * sampling_rate_hz))
step_samples = int(round(step_seconds * sampling_rate_hz))

window_index = 2
start = window_index * step_samples
end = start + window_samples

window_values = leakage_record.values[start:end]

# rectangular window: raw samples
rectangular_values = window_values

# hann window: tapered samples, normalized power
hann = scipy_signal.get_window("hann", window_values.size, fftbins=True).astype(np.float64)
hann /= np.sqrt(np.mean(hann ** 2))
hann_values = window_values * hann

rect_spectrum = fft_magnitude(rectangular_values, sampling_rate_hz=sampling_rate_hz)
hann_spectrum = fft_magnitude(hann_values, sampling_rate_hz=sampling_rate_hz)

plt.figure(figsize=(10, 5))
plt.plot(rect_spectrum.frequencies_hz, rect_spectrum.magnitudes, label="rectangular")
plt.plot(hann_spectrum.frequencies_hz, hann_spectrum.magnitudes, label="hann")
plt.xlabel("Frequency [Hz]")
plt.ylabel("FFT magnitude")
plt.title(f"Window {window_index} FFT magnitude: rectangular vs hann")
#plt.xlim(0, 2000)
plt.legend()
plt.grid(True)
plt.show()

## FFT Magnitude And Dominant Frequency

For a pure sine wave, the dominant FFT magnitude bin should recover the known tone frequency. A two-tone signal should show both designed components.

In [ ]:
from signal_processing_prep.plotting import plot_frequency_spectra
from signal_processing_prep.frequency_domain import psd
import matplotlib.pyplot as plt

for record in [base_sine, noisy_sine, two_tone, burst, sinc]:
    spectrum = fft_magnitude(record)
    print(f"{record.name:28s} dominant_frequency={dominant_frequency(spectrum):7.2f} Hz")

# FFT magnitude
fig_fft, ax_fft = plot_frequency_spectra(
    [base_sine, two_tone, burst, sinc],
    spectrum_type="fft",
    max_frequency_hz=600.0,
)
ax_fft.set_title("FFT Magnitude")



## PSD And Spectral Descriptors

PSD-based descriptors summarize where spectral power sits and how concentrated or noise-like it is.

In [ ]:
spectral_rows = []
for record in records:
    power_spectrum = psd(record, nperseg=2048)
    spectral_rows.append(
        {
            "name": record.name,
            "dominant_frequency_hz": dominant_frequency(power_spectrum),
            "centroid_hz": spectral_centroid(power_spectrum),
            "bandwidth_hz": spectral_bandwidth(power_spectrum),
            "rolloff_85_hz": spectral_rolloff(power_spectrum, rolloff_fraction=0.85),
            "flatness": spectral_flatness(power_spectrum),
        }
    )

spectral_df = pd.DataFrame(spectral_rows)
spectral_df

## Band Energy

Band energy is useful when frequency ranges have physical meaning, such as low-frequency rotation, mid-frequency components, or high-frequency transients.

In [ ]:
bands_hz = {
    "low_0_100_hz": (0.0, 100.0),
    "mid_100_300_hz": (100.0, 300.0),
    "high_300_600_hz": (300.0, 600.0),
}

band_rows = []
for record in records:
    row = {"name": record.name}
    for band_name, (low_hz, high_hz) in bands_hz.items():
        row[band_name] = band_energy(record, low_hz=low_hz, high_hz=high_hz)
    band_rows.append(row)

band_energy_df = pd.DataFrame(band_rows)
band_energy_df

## Time-Frequency Analysis

STFT and spectrograms show how frequency content changes over short windows. Wavelets provide a multi-resolution view. The Hilbert transform is useful for amplitude envelope and instantaneous phase/frequency on narrowband signals.

In [ ]:
chirp = chirp_signal(
    start_frequency_hz=30.0,
    end_frequency_hz=350.0,
    duration_seconds=3.0,
    sampling_rate_hz=4000.0,
    name="30_to_350_hz_chirp",
)

stft_result = stft_analysis(chirp, window_seconds=0.25, step_seconds=0.05, window="hann")
spectrogram_result = spectrogram_analysis(chirp, window_seconds=0.25, step_seconds=0.05, window="hann")
wavelet_result = morlet_wavelet_scalogram(
    chirp,
    min_frequency_hz=20.0,
    max_frequency_hz=400.0,
    n_frequencies=128,
)

stft_result.magnitude.shape, spectrogram_result.power.shape, wavelet_result.power.shape

In [ ]:
plot_time_signal(chirp, start_seconds=0.0, duration_seconds=3.0, max_points=None)

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(10, 7), sharex=True)

axes[0].pcolormesh(
    stft_result.times_seconds,
    stft_result.frequencies_hz,
    stft_result.magnitude,
    shading="auto",
)
axes[0].set_ylim(0.0, 450.0)
axes[0].set_ylabel("Frequency [Hz]")
axes[0].set_title("STFT magnitude")

axes[1].pcolormesh(
    wavelet_result.times_seconds,
    wavelet_result.frequencies_hz,
    wavelet_result.power,
    shading="auto",
)
axes[1].set_ylim(20.0, 400.0)
axes[1].set_ylabel("Frequency [Hz]")
axes[1].set_xlabel("Time [s]")
axes[1].set_title("Morlet wavelet scalogram")

fig.tight_layout()

In [ ]:
# TODO: being able to create the am record with API, so we need DC offset possibility

am_signal = (
    sine_wave(frequency_hz=80.0, duration_seconds=2.0, sampling_rate_hz=1000.0).values
    * (1.0 + 0.5 * sine_wave(frequency_hz=3.0, duration_seconds=2.0, sampling_rate_hz=1000.0).values)
)
am_record = type(base_sine)(
    values=am_signal,
    sampling_rate_hz=1000.0,
    label="amplitude_modulated",
    name="80_hz_carrier_3_hz_envelope",
)
hilbert_result = hilbert_analysis(am_record)

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(am_record.time_seconds, am_record.values, label="signal", alpha=0.7)
ax.plot(hilbert_result.time_seconds, hilbert_result.amplitude_envelope, label="Hilbert envelope")
ax.set_xlim(0.0, 2.0)
ax.set_xlabel("Time [s]")
ax.set_ylabel("Amplitude")
ax.set_title("Hilbert amplitude envelope")
ax.legend()
fig.tight_layout()

## Compare PSD Curves

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(10, 4))
for record in [base_sine, noisy_sine, two_tone, burst, sinc]:
    power_spectrum = psd(record, nperseg=1024)
    ax.semilogy(power_spectrum.frequencies_hz, power_spectrum.power, label=record.name)

ax.set_xlim(0.0, 600.0)
ax.set_xlabel("Frequency [Hz]")
ax.set_ylabel("PSD [amplitude^2 / Hz]")
ax.set_title("Welch PSD comparison")
ax.legend()
fig.tight_layout()


